## 1. Climate Data Visualization and Manipulation in Python (Updated 2025)
Pedro Herrera Lormendez (pedrolormendez@gmail.com) and Conrad Jackisch (conrad.jackisch@tbt.tu-freiberg.de)

**Updates:** Enhanced with improved data validation, error handling, and introduction to modern ECMWF data access tools.

This is the first step to get started with netCDF data in Python. We will use the xarray library to read and visualize climate data with scientific rigor.

### XArray
* **XArray** is focused on N-dimensional arrays of data and its interface is based largely on the netCDF data model (variables, attributes, and dimensions).
* The **DataArray** is one of the basic building blocks of XArray.
* XArray follows the [CF Conventions](http://cfconventions.org/) for climate and forecast metadata.

Quick visualization in the standalone app [Panoply](https://www.giss.nasa.gov/tools/panoply/download/). Open the "sample.nc" sample dataset. Let's explore!

In [1]:
# Importing numpy and xarray
import numpy as np
import xarray as xr
# Other needed tools
import sys
import os
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import warnings
# Assuming your Jupyter notebook is in the 'notebooks' directory
sys.path.append(os.path.abspath('../help_code'))
import tools

# Configure plotting defaults
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore', category=RuntimeWarning)

#### Reading and Validating a netCDF

When working with climate data, it's important to:
1. Verify the data loaded correctly
2. Check units and coordinate systems
3. Validate data ranges for physical plausibility

In [2]:
# Define the file path
file_path = '../data/sample.nc'

# Reading the netCDF file using xarray
try:
    DS = xr.open_dataset(file_path)
    print("✓ Dataset loaded successfully")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# Fixing the longitude coordinates
DS = tools.convert_and_sort_coords(DS)

# Display dataset information
DS

✓ Dataset loaded successfully


AttributeError: module 'tools' has no attribute 'convert_and_sort_coords'

In [ ]:
# Validate the dataset structure
print("Dataset Variables:")
for var in DS.data_vars:
    print(f"  {var}: {DS[var].attrs.get('long_name', 'No description')}")
    print(f"    Units: {DS[var].attrs.get('units', 'Not specified')}")
    print(f"    Shape: {DS[var].shape}")
    print()

#### Extracting Variables with Unit Awareness

In [ ]:
# Extracting the two-metre temperature variable (t2m)
t2m = DS.t2m

# Check current units
original_units = t2m.attrs.get('units', 'unknown')
print(f"Original units: {original_units}")

t2m

In [ ]:
# Checking the time coordinate values
print(f"Time range: {t2m.time.values[0]} to {t2m.time.values[-1]}")
print(f"Number of time steps: {len(t2m.time)}")
print(f"Temporal resolution: {pd.to_datetime(t2m.time.values[1]) - pd.to_datetime(t2m.time.values[0])}")
t2m.time

#### Unit Conversion with Validation

**Physical constraints for 2m temperature:**
* Valid range in Kelvin: approximately 180-330 K (most Earth surface conditions)
* Valid range in Celsius: approximately -90 to +60 °C

Always update metadata attributes after unit conversions to maintain data integrity.

In [ ]:
# Convert t2m to °C with validation
if original_units == 'K':
    # Validate data is in expected range before conversion
    t_min, t_max = t2m.min().values, t2m.max().values
    print(f"Temperature range before conversion: {t_min:.2f} to {t_max:.2f} K")
    
    if t_min < 150 or t_max > 350:
        warnings.warn(f"Temperature values outside expected range: {t_min:.2f} to {t_max:.2f} K")
    
    # Convert to Celsius
    t2m = t2m - 273.15
    
    # Update units attribute to maintain metadata integrity
    t2m.attrs['units'] = '°C'
    print("✓ Converted to Celsius and updated metadata")
else:
    print(f"Data already in {original_units}, no conversion needed")

In [ ]:
# Checking the minimum and maximum values
t_min_c = t2m.min().values
t_max_c = t2m.max().values
print(f"The minimum temperature is {t_min_c:.2f}°C")
print(f"The maximum temperature is {t_max_c:.2f}°C")

# Validate physical plausibility
if t_min_c < -90 or t_max_c > 60:
    warnings.warn(f"Temperature values outside typical Earth range: {t_min_c:.2f} to {t_max_c:.2f}°C")

In [ ]:
# Quickly visualizing the data
# Printing the dimensions of the data
print(f"Dimensions: {t2m.dims}")
print(f"Shape: {t2m.shape}")

# Accessing the first time dimension [0,:,:] - 0 for first dimension, -1 for the last
t2m[0].plot(cmap='RdBu_r', cbar_kwargs={'label': 'Temperature (°C)'})
plt.title(f'2m Temperature at {str(t2m.time[0].values)[:19]}')

### Analyzing the Extreme Temperature Event on 19-07-2022

**Context:** July 2022 saw record-breaking heatwaves across Europe, with the UK recording temperatures above 40°C for the first time.

* Extracting the data at the time point of interest: 19-07-2022 15:00

In [ ]:
# Extreme temperature on 19-07 at 15:00 hrs
t2m_20220719 = t2m.sel(time='2022-07-19T15:00:00.000000000')
t2m_20220719

In [ ]:
# Plotting the data on 19-07-2022 at 15:00 hrs
t2m_20220719.plot(cmap='RdYlBu_r', cbar_kwargs={'label': 'Temperature (°C)'})
plt.title('2m Temperature - July 19, 2022 at 15:00 UTC')

#### Enhanced Cartographic Visualization

Using Cartopy for professional map projections with coastlines, borders, and geographic context.

In [ ]:
# Enhanced plot with coastlines and geographic features
fig = plt.figure(figsize=[13, 5])
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=0))

# Plot data
im = t2m_20220719.plot(ax=ax, transform=ccrs.PlateCarree(), 
                       cmap='RdYlBu_r', 
                       cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8})

# Add geographic features
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)

plt.title('2m Temperature - July 19, 2022 at 15:00 UTC', fontsize=12, fontweight='bold')

In [ ]:
# Alternative projection: Orthographic (globe view)
fig = plt.figure(figsize=[10, 5])
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Orthographic(central_longitude=20, central_latitude=45))

t2m_20220719.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(), 
                              cmap='RdYlBu_r',
                              extend='both',
                              cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.7})

ax.coastlines(linewidth=0.8)
ax.gridlines(linewidth=0.5, alpha=0.5)
plt.title('2m Temperature - Orthographic Projection', fontsize=12, fontweight='bold')

#### Time Series Extraction at Specific Locations

**Note on nearest-neighbor selection:** When using `.sel()` with exact coordinates, xarray finds the nearest grid point. For precise location matching, use `method='nearest'` explicitly.

In [ ]:
# Extract at lon=50°E and lat=41°N (approximate location in Caspian region)
t2m_series = t2m.sel(lon=50, lat=41, method='nearest')

# Display actual coordinates selected
print(f"Selected coordinates: {t2m_series.lon.values}°E, {t2m_series.lat.values}°N")

plt.figure(figsize=[10, 4])
t2m_series.plot()
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Time Series at {t2m_series.lat.values}°N, {t2m_series.lon.values}°E')
plt.grid(True, alpha=0.3)

In [ ]:
# London: approximately 51°N and 0°E
t2m_london = t2m.sel(lon=0, lat=51, method='nearest')

print(f"London grid point: {t2m_london.lat.values}°N, {t2m_london.lon.values}°E")

plt.figure(figsize=[10, 4])
t2m_london.plot(color='darkblue')
plt.ylabel('Temperature (°C)')
plt.title('Temperature Time Series - London Area')
plt.grid(True, alpha=0.3)

In [ ]:
# Plotting both time series together with proper labels
plt.figure(figsize=[12, 5])
t2m_series.plot(label=f'Location 1 ({t2m_series.lat.values}°N, {t2m_series.lon.values}°E)')
t2m_london.plot(label=f'London ({t2m_london.lat.values}°N, {t2m_london.lon.values}°E)')
plt.ylabel('Temperature (°C)')
plt.title('Temperature Comparison - Two Locations')
plt.legend()
plt.grid(True, alpha=0.3)

#### Spatial Subsetting

Extracting regional data for focused analysis. Using `slice()` for coordinate ranges.

In [ ]:
# Zoom over Europe: latitude 35 to 70°N and longitude 0 to 40°E
# .sel function: DataArray.sel(lat=slice(y1, y2), lon=slice(x1, x2))
t2m_EU = t2m_20220719.sel(lat=slice(35, 70), lon=slice(0, 40))

print(f"European subset shape: {t2m_EU.shape}")
print(f"Lat range: {t2m_EU.lat.min().values}°N to {t2m_EU.lat.max().values}°N")
print(f"Lon range: {t2m_EU.lon.min().values}°E to {t2m_EU.lon.max().values}°E")

t2m_EU.plot(cmap='RdBu_r', cbar_kwargs={'label': 'Temperature (°C)'})
plt.title('European Region - July 19, 2022 Heatwave')

In [ ]:
# Enhanced European map with geographic features
fig = plt.figure(figsize=[10, 7])
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

t2m_EU.plot(ax=ax, transform=ccrs.PlateCarree(), 
            cmap='RdBu_r',
            cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8})

ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)
ax.set_extent([0, 40, 35, 70], crs=ccrs.PlateCarree())

plt.title('2m Temperature over Europe - July 19, 2022 at 15:00 UTC', fontsize=12, fontweight='bold')

In [ ]:
# Exploring the longitude values (demonstrating coordinate conversion)
print("First 20 longitude values:")
print(t2m_20220719.lon.values[:20])

In [ ]:
# North America: lat = 25 to 70°N and lon = -120 to -60°W
t2m_20220719_converted = tools.convert_and_sort_coords(t2m_20220719)
t2m_NA = t2m_20220719_converted.sel(lat=slice(20, 70), lon=slice(-130, -60))

print(f"North America subset shape: {t2m_NA.shape}")
t2m_NA.plot(cmap='RdBu_r', cbar_kwargs={'label': 'Temperature (°C)'})
plt.title('North America - July 19, 2022 at 15:00 UTC')

In [ ]:
# Enhanced North America map
fig = plt.figure(figsize=[10, 6])
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

t2m_NA.plot(ax=ax, transform=ccrs.PlateCarree(), 
            cmap='RdBu_r',
            cbar_kwargs={'label': 'Temperature (°C)'})

ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)
ax.set_extent([-130, -60, 20, 70], crs=ccrs.PlateCarree())

plt.title('2m Temperature over North America', fontsize=12, fontweight='bold')

### Basic Statistics, Data Summarization, and Aggregation

XArray provides powerful methods for computing statistics along specific dimensions.

In [ ]:
# Computing the mean value across all dimensions (time, lat, lon)
print(f"The t2m variable has {t2m.shape} dimensions (time, lat, lon)")
global_mean = t2m.mean()
print(f"The global mean temperature for this period is: {global_mean.values:.2f}°C")

#### DataArray.mean() - Dimension-Specific Averaging

**Important:** When computing spatial means of temperature, consider:
* Area-weighting for accurate global/regional averages (not shown here, but see Notebook 4)
* Different temporal averaging periods (hourly → daily → monthly)

In [ ]:
# Computing the mean value across spatial dimensions (lat, lon)
# This creates a time series of globally-averaged temperature
mean_1d = t2m.mean(dim=('lat', 'lon'))
print(f"The spatial mean has shape {mean_1d.shape} (time series)")

plt.figure(figsize=[12, 5])
mean_1d.plot()
plt.ylabel('Temperature (°C)')
plt.title('Spatially-Averaged Temperature Time Series')
plt.grid(True, alpha=0.3)

In [ ]:
# Computing the mean value across the time dimension
# This creates a 2D map of time-averaged temperature
mean_2d = t2m.mean(dim='time')
print(f"The temporal mean has shape {mean_2d.shape} (lat, lon)")

mean_2d.plot(cmap='RdBu_r', cbar_kwargs={'label': 'Mean Temperature (°C)'})
plt.title('Time-Averaged Temperature (July 17-20, 2022)')

#### DataArray.rolling() - Moving Window Smoothing

**Statistical note:** Rolling means reduce high-frequency noise but introduce temporal correlation. The `center=True` parameter centers the window on each point.

In [ ]:
plt.figure(figsize=[12, 5])

# Original hourly time series
t2m_london.plot(label='Hourly', alpha=0.6, linewidth=1)

# 12-hour rolling mean
t2m_london.rolling(time=12, center=True).mean().plot(label='12-hour rolling mean', linewidth=2)

# 24-hour rolling mean (daily smoothing)
t2m_london.rolling(time=24, center=True).mean().plot(label='24-hour rolling mean', linewidth=2)

plt.ylabel('Temperature (°C)')
plt.title('Temperature with Rolling Means - London')
plt.legend()
plt.grid(True, alpha=0.3)

#### DataArray.resample() - Temporal Aggregation

**Difference from rolling():** `resample()` downsamples to a coarser frequency, creating non-overlapping periods. Commonly used for hourly → daily, daily → monthly aggregations.

In [ ]:
# Resample to daily means
t2m_series_daily = t2m_series.resample(time='1D').mean()

plt.figure(figsize=[12, 5])
t2m_series.plot(label='Hourly', alpha=0.6)
t2m_series_daily.plot(marker='o', markersize=8, label='Daily mean', linewidth=2)

plt.ylabel('Temperature (°C)')
plt.title('Hourly vs Daily Mean Temperature')
plt.legend()
plt.grid(True, alpha=0.3)

#### DataArray.sum() - Accumulation Operations

**Precipitation units:** ERA5 total precipitation is in meters. Converting to mm (1 m = 1000 mm).

In [ ]:
# Reading the precipitation "tp" variable
tp = DS.tp
print(f"Original units: {tp.attrs.get('units', 'Not specified')}")

# Converting to mm (multiply by 1000)
tp = tp * 1000
tp.attrs['units'] = 'mm'

# Computing the total precipitation along the time dimension
tp_sum = tp.sum(dim='time')
print(f"The tp_sum variable has dimensions {tp_sum.dims} with shape {tp_sum.shape}")
print(f"Total accumulated precipitation range: {tp_sum.min().values:.1f} to {tp_sum.max().values:.1f} mm")

In [ ]:
# Plotting the total accumulated precipitation
tp_sum.plot(cmap='Blues', cbar_kwargs={'label': 'Total Precipitation (mm)'})
plt.title('Total Precipitation - July 17-20, 2022')

#### Practice Time
<div style="background-color:lightgreen; padding:10px">
    Plot the time series of the t2m variable for 3 different locations.
    <ul>
        <li>The three plots should be combined into a single graph</li>
        <li>Add a different color to each line plot</li>
        <li>Include proper labels and a legend</li>
        <li>Add grid lines for better readability</li>
    </ul>
</div>

In [ ]:
# Your code goes here
# Hint: Use .sel(lon=X, lat=Y, method='nearest') for each location
# Example locations: Madrid (40°N, -4°E), Berlin (52°N, 13°E), Athens (38°N, 24°E)


#### Practice Time
<div style="background-color:lightgreen; padding:10px">
    Plot the hourly and daily behavior of temperature and precipitation over a given lat and lon point.
    <ul>
        <li>Extract the t2m and tp over a single lat, lon point</li>
        <li>Plot the hourly time series of the t2m and tp of the selected point</li>
        <li>Compute the daily mean of t2m and the daily accumulated rainfall of tp of the selected point</li>
        <li>Plot the daily mean of t2m and daily rainfall of variable tp</li>
        <li>Add appropriate labels and units</li>        
    </ul>
</div>

In [ ]:
# Your code goes here
# Remember: use .resample(time='1D').mean() for temperature
#           use .resample(time='1D').sum() for precipitation


### Introduction to Modern Data Access Tools

**Looking ahead:** In Notebook 3, we'll explore how to download climate data using the new **ecmwf-datastores-client**, which provides:
* Unified interface for multiple ECMWF data stores (CDS, ADS)
* Enhanced job tracking and management
* Asynchronous data retrieval
* Better error handling and validation

The legacy `cdsapi` is still supported, but the new client offers improved functionality for accessing ERA5, CMIP6, and other datasets.

---
### Summary

**What we learned:**
1. Loading and validating netCDF climate data with XArray
2. Proper unit conversions with metadata updates
3. Physical validation of climate data
4. Spatial subsetting and time series extraction
5. Statistical operations: mean, rolling, resample, sum
6. Professional cartographic visualization with Cartopy

**Next steps:**
* Notebook 2: Advanced time series analysis and trend detection
* Notebook 3: Data access with modern ECMWF tools
* Notebook 4: Climatologies and anomaly detection

**Key takeaways for scientific computing:**
* Always validate units and coordinate systems
* Update metadata attributes after transformations
* Check physical plausibility of results
* Use appropriate statistical methods for climate data